In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

In [16]:
df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- pagina: integer (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+------------------+--------------------+------+-----+
|                 _id|categoria|      fecha_captura|             grupo|                item|pagina|valor|
+--------------------+---------+-------------------+------------------+--------------------+------+-----+
|69dd904a53b9d5371...|   travel|2026-04-14 00:54:33|Mercado_Laboral_TI|It's Only the Him...|     1|45.17|
|69dd904a53b9d5371...|   travel|2026-04-14 00:54:33|Mercado_Laboral_TI|Full Moon over No...|     1|49.43|
|69dd904a53b9d5371...|   travel|2026-04-14 00:54:33|Mercado_Laboral_TI|See America: A Ce...|     1|48.87|
|69dd904a53b9d5371...|   travel|2026-04-14 00:54:33|Mercado_Laboral_TI|Vagabonding: An U

In [17]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 11
Registros validos: 11


In [18]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|It's Only the Him...|45.17|
|Full Moon over No...|49.43|
|See America: A Ce...|48.87|
|Vagabonding: An U...|36.94|
|Under the Tuscan Sun|37.33|
|  A Summer In Europe|44.34|
|The Great Railway...|30.54|
|A Year in Provenc...|56.88|
|The Road to Littl...|23.21|
|Neither Here nor ...|38.95|
|1,000 Places to S...|26.08|
+--------------------+-----+



In [20]:
df.filter(df["valor"] > 50).show()

+--------------------+---------+-------------------+------------------+--------------------+------+-----+
|                 _id|categoria|      fecha_captura|             grupo|                item|pagina|valor|
+--------------------+---------+-------------------+------------------+--------------------+------+-----+
|69dd904a53b9d5371...|   travel|2026-04-14 00:54:34|Mercado_Laboral_TI|A Year in Provenc...|     1|56.88|
+--------------------+---------+-------------------+------------------+--------------------+------+-----+



In [21]:
df.groupBy("grupo").count().show()

+------------------+-----+
|             grupo|count|
+------------------+-----+
|Mercado_Laboral_TI|   11|
+------------------+-----+



In [22]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+-----------------+-------------+-------------+
|categoria|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+-----------------+-------------+-------------+
|   travel|             11|39.79454545454546|        23.21|        56.88|
+---------+---------------+-----------------+-------------+-------------+



In [23]:
fila = df.orderBy(F.desc("valor")).select(
    "item", "categoria", "fecha_captura", "valor"
).first()

print("Producto más caro:", fila["item"])
print("Categoría:", fila["categoria"])
print("Fecha y hora de captura:", fila["fecha_captura"])
print("Valor:", fila["valor"])

Producto más caro: A Year in Provence (Provence #1)
Categoría: travel
Fecha y hora de captura: 2026-04-14 00:54:34
Valor: 56.88


In [28]:
from pyspark.sql import functions as F

NOMBRE_GRUPO = "Mercado_Laboral_TI"

ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: Mercado_Laboral_TI ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
|   travel|          11|       39.79|  2026-04-14 00:54:34|
+---------+------------+------------+---------------------+

